In [ ]:
%load_ext autoreload
%autoreload 2
import os

os.chdir("..")
print(os.getcwd())

In [ ]:
import re

import pandas as pd
import requests
import wandb

In [ ]:
def pprint(dictionary):
    for k, v in dictionary.items():
        print(k, ":", v)

In [ ]:
os.makedirs("outputs/alignment", exist_ok=True)

# Table prep functions

In [ ]:
def format_mean_sem(mean, sem, decimals=3, scale=1.0, math_mode=True):
    """Format a single mean/SEM pair as a LaTeX string, with optional scaling."""
    if pd.isna(mean):
        return "--"
    m = mean * scale
    if pd.isna(sem):
        s = f"{m:.{decimals}f}"
    else:
        s = f"{m:.{decimals}f} $\\pm$ {sem * scale:.{decimals}f}"
    return f"{s}" if math_mode else s


def add_mean_sem_columns(df, metrics, math_mode=True):
    """
    metrics: dict mapping output_col_name -> dict with:
        mean_col, sem_col, decimals (int), scale (float, default 1.0)
    """
    df = df.copy()
    for out_col, spec in metrics.items():
        df[out_col] = [
            format_mean_sem(
                m,
                s,
                decimals=spec.get("decimals", 3),
                scale=spec.get("scale", 1.0),
                math_mode=math_mode,
            )
            for m, s in zip(df[spec["mean_col"]], df[spec["sem_col"]])
        ]
    return df


def get_latex(df, columns=None, escape=False):
    if columns is not None:
        df = df[columns]
    else:
        columns = list(df.columns)

    latex_table = df.to_latex(
        index=False,
        columns=columns,
        escape=escape,  # False: let \pm and $ pass through untouched
    )
    print(latex_table)

    payload = {
        "formula": latex_table,
        "fsize": "54px",
        "fcolor": "000000",
        "mode": "0",
        "out": "1",
        "remhost": "quicklatex.com",
        "preamble": r"\usepackage{booktabs}\usepackage{amsmath}",
    }
    response = requests.post("https://quicklatex.com/latex3.f", data=payload)
    print(response.text)
    return latex_table


def parse_modality(x, modalities):
    for k, v in modalities.items():
        if k in x:
            return v

# WANDB API parsing

In [ ]:
# Connect to wandb api
api = wandb.Api()
entity = "aether_xai"

In [ ]:
project = "satbird-usa-summer_alignment"
results_df_path = "outputs/alignment/results_s2bms.csv"

## csv log

In [ ]:
df = None

# Separate concept columns

In [ ]:
os.getcwd()

In [ ]:
import json

# Collect columns into 3 categories per version
versions = {"v4": {}}

for k, v in versions.items():
    with open(f"data/satbird-USA-summer/concept_captions/{k}.json", "r") as f:
        data = json.load(f)

    v["in_dist_valid"] = []
    v["in_dist_unvalid"] = []

    for d in data:
        if "test" in d["col"]:
            name = d["col"].replace("test-", "test_dyn_k_index_")
            name = f"{name.replace('aux_', '')}_{'max' if d['is_max'] else 'min'}"
            v["out_dist"].append(name)
        else:
            name = f"{d['col'].replace('aux_', 'test_dyn_k_index_')}_{'max' if d['is_max'] else 'min'}"
            if d.get("val_av", True):
                v["in_dist_valid"].append(name)
            else:
                v["in_dist_unvalid"].append(name)

In [ ]:
versions["v4"]

## fetching

In [ ]:
def log_concept_avr(run, cols, name):
    metrics = dict(run.summary)
    matched_metrics = [metrics[c] for c in cols]
    value = sum(matched_metrics) / len(matched_metrics)
    run.summary.update({name: value})
    print(name, value)


def flatten(d: dict, parent_key: str = "", sep: str = ".") -> dict:
    """Flatten a nested config dict (e.g. a Hydra config) into dot-separated keys."""
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items

In [ ]:
projects = ["satbird-usa-summer_alignment"]

In [ ]:
api = wandb.Api()
for project in projects:
    runs_iterator = api.runs(f"{entity}/{project}", order="+created_at")
    run_list = []

    for i, run in enumerate(runs_iterator):
        if run.state != "finished":
            print(f"{run.id} is not finished.")
            continue
        concept_v = run.config["data"]["caption_builder"]["concepts_fname"].replace(".json", "")
        if concept_v == "${version}":
            concept_v = run.summary["experiment"]
            match = re.search(r"(?<=_)v\d+", concept_v)
            concept_v = match.group() if match else None

        if run.summary.get("test_avr_top-dyn_k_index_in_dist_valid") is None:
            try:
                in_dist_valid = versions[concept_v]["in_dist_valid"]
                in_dist_unvalid = versions[concept_v]["in_dist_unvalid"]

                log_concept_avr(run, in_dist_valid, "test_avr_top-dyn_k_index_in_dist_valid")
                log_concept_avr(run, in_dist_unvalid, "test_avr_top-dyn_k_index_in_dist_unvalid")
            except Exception as e:
                if concept_v == "v8":
                    print(f"v8 issue {run.id}: {e}")
                else:
                    print(f"{run.id}: {e}")

        if df is not None:
            if run.id in list(df.run_id):
                print(
                    f'{run.summary["experiment"]} with seed={run.config["seed"]} already in results.'
                )
                continue

        temp = dict(run.summary)
        config_dict = flatten(run.config)
        captures = {}
        captures["experiment"] = run.summary["experiment"]
        captures["run_id"] = run.id
        for k, v in temp.items():
            # if "test" not in k and "best" not in k:
            #     captures[f"best_{k}"] = v
            # else:
            #     captures[k] = v
            if "test" in k and "dyn_k_index" in k:
                captures[k] = v

        captures = {**captures, **config_dict}
        del captures["tags"]
        run_list.append(captures)
        print(f'{run.summary["experiment"]} with seed={run.config["seed"]} logged.')

In [ ]:
list_aef = []
list_other = []

for r in run_list:
    # if r['experiment'].startswith('aef'):
    #     list_aef.append(r)
    # elif r['experiment'] in ['higher', 'lower']:
    #     continue
    # else:
    #     list_other.append(r)
    if r["experiment"].startswith("baseline"):
        list_other.append(r)
    else:
        list_aef.append(r)
print(f"len(list_aef)={len(list_aef)}, len(list_other)={len(list_other)}")
df_dict = {}
df_dict["aef"] = pd.DataFrame(list_aef)
if len(list_other) > 0:
    df_dict["baseline"] = pd.DataFrame(list_other)

import re

for k, v in df_dict.items():
    test_cols = [c for c in v.columns if "test" in c and "dyn_k_index" in c]

    dict_cols = {}

    dict_cols["bioclim"] = [c for c in test_cols if "_bio_" in c]
    dict_cols["dw_all"] = [c for c in test_cols if "_dw_" in c]
    soil_vars = ["bdticm", "cecsol", "clyppt", "orcdrc", "phihox", "sltppt", "sndppt"]
    dict_cols["soil"] = [c for c in test_cols if any([s in c for s in soil_vars])]

    cols_groups = []
    for k2, v2 in dict_cols.items():
        new_col = f"test_avr_top-dyn_k_index_aggr_{k2}"
        df_dict[k][new_col] = df_dict[k][v2].mean(axis=1)
        cols_groups.append(new_col)
    df_dict[k]["group_av"] = df_dict[k][cols_groups].mean(axis=1)

    cols_drop = []

    for c in v.columns:
        if any([isinstance(x, list) for x in v[c]]):
            cols_drop.append(c)
        elif v[c].nunique() <= 1:
            cols_drop.append(c)

    df_dict[k].drop(columns=cols_drop, inplace=True)

In [ ]:
df_dict["aef"].sort_values(
    by="test_avr_top-dyn_k_index_in_dist_valid", ascending=False, inplace=True
)
df_dict["aef"]

# AEF

In [ ]:
best_aef_run_config = df_dict["aef"][df_dict["aef"]["experiment"] == "s0.1"]
assert len(best_aef_run_config) == 3, "There should be only one best AEF run."

In [ ]:
best_aef_run_config["model.geo_encoder._target_"]  # = df_dict['aef']

# BASELINE

In [ ]:
df_dict["baseline"][
    df_dict["baseline"]["model._target_"] == "src.models.text_alignment_model.TextAlignmentModel"
]

In [ ]:
save_table = True
runs_df = df_dict["baseline"].copy()

cols_params = [
    c
    for c in runs_df.columns
    if "index" not in c
    and c
    not in ["experiment", "run_id", "model/params/total", "model/params/trainable", "group_av"]
]
save_tables = False
criterion_metric = "test_avr_top-dyn_k_index_in_dist_valid"
metrics_table = [
    "test_avr_top-dyn_k_index_aggr_bioclim",
    "test_avr_top-dyn_k_index_aggr_dw_all",
    "test_avr_top-dyn_k_index_aggr_soil",
]
metrics_use = metrics_table + [criterion_metric]
columns_name_map = {
    "name_plot": "Model",
    "test_avr_top-dyn_k_index_aggr_bioclim": "Biocl.",
    "test_avr_top-dyn_k_index_aggr_dw_all": "DW LC",
    "test_avr_top-dyn_k_index_aggr_soil": "Soil",
    "average": "Av.",
    "test_avr_top-dyn_k_index_in_dist_valid": "Av. (sub)",
    "data.dataset.modalities.s2.preprocessing": "S2 preprocess.",
    "model.model_name": "Model name",
    "data.dataset.modalities.s2.channels": "Chan.",
}
model_name_map = {
    "AEF_ablations": {
        "AEF labelled, AEF proj.": "AEF, labelled, AEF proj., sep.",
        "AEF unlabelled, AEF proj.": "AEF, unlabelled, AEF proj., sep.",
        "AEF unlabelled, text proj.": "AEF, unlabelled, text proj., sep.",
        "AEF unlabelled, both proj.": "AEF, unlabelled, both proj., sep.",
        "AEF unlabelled, AEF proj., mixed": "AEF, unlabelled, AEF proj., mix.",
        "AEF labelled, AEF proj., mixed": "AEF, labelled, AEF proj., mix.",
    },
    "main": {
        "AEF": "AEF, separate captions",
        "ClipImgEncoder": "CLIP",
        "DOFACLIP_model": "DOFA-CLIP",
        "GeoClipCoordinateEncoder": "GeoCLIP",
        "GeoRSCLIP_model": "GeoRSCLIP",
        "RemoteCLIP_model": "RemoteCLIP",
        "SkyCLIP_model": "SkyCLIP",
    },
}
assert len(best_aef_run_config) == 3, "There should be only one best AEF run."

best_aef_run_config["name_plot"] = "AEF"
runs_sel = (
    best_aef_run_config.groupby("name_plot")
    .agg({k: ["mean", "sem"] for k in metrics_use})
    .reset_index()
)
runs_sel["name_plot"] = "AEF"
df_best_runs = runs_sel.copy()  # Ensure the best_run has the same columns as df_best_runs

for metric in metrics_use:
    mean_col = (metric, "mean")
    sem_col = (metric, "sem")
    tmp = [
        format_mean_sem(m, s, decimals=2, scale=1.0, math_mode=True)
        for m, s in zip(df_best_runs[mean_col], df_best_runs[sem_col])
    ]
    df_best_runs.drop(columns=[mean_col, sem_col], inplace=True)
    if metric == "test_avr_top-dyn_k_index_aggr_topo":
        continue
    df_best_runs[metric] = tmp

df_best_runs.columns = [c[0] if isinstance(c, tuple) else c for c in df_best_runs.columns]

## get best baseline model for each combination of model and geo_encoder, and compare to best AEF model
runs_df = df_dict["baseline"].copy()

cols_params = [
    c
    for c in runs_df.columns
    if "index" not in c
    and c not in ["experiment", "run_id", "model/params/total", "model/params/trainable"]
]
# cols_main = ['model._target_', 'model.geo_encoder._target_']
# unique_combinations = runs_df[cols_main].drop_duplicates()
# assert False
runs_df["name_plot"] = runs_df["model._target_"].apply(lambda x: x.split(".")[-1].lstrip("build_"))
if "TextAlignmentModel" in runs_df["name_plot"].values:
    runs_df["name_plot"][runs_df["name_plot"] == "TextAlignmentModel"] = "CLIP"
for name_combi in runs_df["name_plot"].unique():
    runs_sel = runs_df[runs_df["name_plot"] == name_combi].copy()
    runs_sel["average"] = runs_sel[metrics_table].mean(axis=1)
    best_run = runs_sel.loc[runs_sel[(criterion_metric)].idxmax()]
    best_run = best_run[
        df_best_runs.columns
    ]  # Ensure the best_run has the same columns as df_best_runs
    assert all(
        best_run.index == df_best_runs.columns
    ), f"Column mismatch: {best_run.index} vs {df_best_runs.columns}"
    df_best_runs = pd.concat([df_best_runs, pd.DataFrame([best_run])], ignore_index=True)


## Select different model name maps for different tables
for name_map, model_map in model_name_map.items():
    if name_map != "main":
        continue
    df_best_runs_sel = df_best_runs.copy()
    df_best_runs_sel = df_best_runs_sel[
        df_best_runs_sel["name_plot"].isin(model_map.keys())
    ].copy()
    df_best_runs_sel["name_plot"] = df_best_runs_sel["name_plot"].map(model_map)
    df_best_runs_sel.rename(columns=columns_name_map, inplace=True)

    for c in df_best_runs_sel.columns:
        if c == "Model":
            continue
        df_best_runs_sel[c] = df_best_runs_sel[c].apply(
            lambda x: f"{x:.2f}" if isinstance(x, float) else x
        )

    def get_float_from_str(df, col_ignore=["Model"]):
        df_float = df.copy()
        for c in df_float.columns:
            if c in col_ignore:
                continue
            df_float[c] = df_float[c].apply(
                lambda x: float(x.split(" ")[0]) if isinstance(x, str) and " " in x else float(x)
            )
        return df_float

    df_best_runs_float = get_float_from_str(df_best_runs_sel, col_ignore=["Model"])
    inds_sorted = df_best_runs_float["Av. (sub)"].sort_values(ascending=True).index
    ## Put AEF last (index 0):
    inds_sorted = [i for i in inds_sorted if i != 0] + [0]
    df_best_runs_sel = df_best_runs_sel.loc[inds_sorted].reset_index(drop=True)
    df_best_runs_float = df_best_runs_float.loc[inds_sorted].reset_index(drop=True)

    ## best model per column:
    for c in df_best_runs_float.columns:
        if c == "Model":
            continue
        best_val = df_best_runs_float[c].max()
        inds_best = df_best_runs_float[df_best_runs_float[c] == best_val].index
        df_best_runs_sel.loc[inds_best, c] = df_best_runs_sel.loc[inds_best, c].apply(
            lambda x: f"\\textbf{{{x}}}" if isinstance(x, str) else f"\\textbf{{{x:.2f}}}"
        )

    if name_map == "main":
        caption = "\\textbf{Retrieval scores of the AlphaEarth (AEF) alignment models on the SatBird task.} Scores show the average retrieval $I_{\\text{top-}k_l}$ score for the bioclimatic, Dynamic World (DW) land cover, and soil groups, as well as the average across a subset (Av.\\ (sub)) of columns (used as validation criterion)."
    # elif name_map == 'AEF_ablations':
    #     caption = "\\textbf{Ablation study of the AlphaEarth alignment model on S2BMS.} The projection network (AEF-only, text-only, both), data (labelled, unlabelled), and caption strategy (separate, mixed) were varied."

    if save_table:
        df_best_runs_sel.to_latex(
            f"/Users/tplas/repos/ms_aether_biodiv/tables/alignment_satbird_summary_{name_map}.tex",
            index=False,
            escape=False,
            label=f"tab:align_satbird_summary_{name_map}",
            caption=caption,
        )

In [ ]:
df_best_runs_sel

In [ ]:
df_dict["baseline"].sort_values(
    by="test_avr_top-dyn_k_index_in_dist_valid", ascending=False, inplace=True
)
df_dict["baseline"]

In [ ]:
soil_vars = ["bdticm", "cecsol", "clyppt", "orcdrc", "phihox", "sltppt", "sndppt"]
len(soil_vars)

## Previous:


In [ ]:
cols_of_interest = {
    "experiment": "Modality (best config.)",
    "test_avr_top-dyn_k_index_in_dist_valid": "In distribution (present)",
    "test_avr_top-dyn_k_index_in_dist_unvalid": "In distribution (absent)",
}

In [ ]:
sub_runs_df = runs_df[cols_of_interest.keys()]

grouped = sub_runs_df.groupby("experiment")

# mean and SEM in one go
summary_mean = grouped.mean(numeric_only=True)
summary_sem = grouped.sem(numeric_only=True)  # pandas has this built in: std/sqrt(n)

# also useful to know how many runs went into each SEM
summary_n = grouped.size().rename("n_runs")

# rename sem columns so they don't clash with mean columns
summary_sem = summary_sem.rename(columns={c: f"{c}_sem" for c in summary_sem.columns})

summary = summary_mean.join(summary_sem).join(summary_n)
summary = summary.sort_values("test_avr_top-dyn_k_index_in_dist_valid", ascending=False)
summary.reset_index(inplace=True)

In [ ]:
summary